# Week 6: Stress-testing a sentiment model
AI for Good, Minor ADSAI. Individual assignment (section 4 of the workshop notebook).

**My context: passenger feedback for a railway company.** I chose it because I am already
reading thousands of train-related posts in my Hackathon 6 project (Bahn-Puls), so I know what
real passenger messages look like.

## The 60-second version (how I explain it out loud)
1. A company wants a ready-made model to read passenger feedback and say "positive" or "negative".
2. I am the **tester**, not the builder. I wrote 14 realistic passenger sentences and decided
   **by myself first** what a person would say.
3. Then I ran the model (DistilBERT, trained on English movie reviews) and compared.
4. It got **9 of 14** right. The 5 it got wrong are the ones that matter: sarcasm, slang with typos,
   a long polite complaint, and German or Dutch.
5. It was **confidently** wrong (94% to 99% sure), so you cannot use its confidence as a warning light.
6. My answer: use it only as a first sorting step, in English, with a human checking a sample.

## 0. Setup
This loads two small ready-made models. On Google Colab run `%pip install -q transformers torch pandas`
first. Nothing is trained here, I only use the models.

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# Model 1: reads a sentence and says POSITIVE or NEGATIVE (DistilBERT, English only)
sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Model 2: guesses the next word (DistilGPT-2), used for the bias probe
lm_tok = AutoTokenizer.from_pretrained("distilgpt2")
lm = AutoModelForCausalLM.from_pretrained("distilgpt2").eval()

def show(sentences):
    for result, sentence in zip(sentiment(sentences), sentences):
        print(result["label"], round(result["score"], 2), "|", sentence)

def next_words(prompt, k=6):
    # the k most likely next words and their probability
    ids = lm_tok(prompt, return_tensors="pt")
    with torch.no_grad():
        p = torch.softmax(lm(**ids).logits[0, -1], -1)
    top = p.topk(k)
    return [(lm_tok.decode(int(i)).strip(), round(float(x), 3)) for x, i in zip(top.values, top.indices)]

def pronoun_odds(prompt):
    # probability of " he" and " she" as the next word
    ids = lm_tok(prompt, return_tensors="pt")
    with torch.no_grad():
        p = torch.softmax(lm(**ids).logits[0, -1], -1)
    return {w: round(float(p[lm_tok.encode(" " + w)[0]]), 3) for w in ["he", "she"]}

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6277.18it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:  38%|███▊      | 29/76 [00:00<00:00, 183.56it/s]

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 474.18it/s]

## 1. Context I chose
**Company:** a railway operator (like Deutsche Bahn or NS) that wants to read passenger feedback
automatically instead of by hand.

**Who is at risk if the model is wrong:** passengers whose complaint is not seen, and passengers
who write in a way the model was not trained on (other languages, slang, irony).

## 2. My test sentences and my own labels (written BEFORE running the model)
14 sentences. I covered all six kinds from the assignment: negation, mixed feelings, sarcasm or irony,
a long sentence where the key word is far from the end, typos or slang, and other languages.
The label is what a normal person would say about the passenger's experience.

In [2]:
tests = [
    # (sentence, kind, my label)
    ("The train arrived on time and the staff were friendly.", "plain", "POSITIVE"),
    ("The train was not late today.", "negation", "POSITIVE"),
    ("The train was not good at all.", "negation", "NEGATIVE"),
    ("The seats were comfortable, but the train was forty minutes late and nobody told us why.", "mixed feelings", "NEGATIVE"),
    ("Oh great, another cancelled train. Just what I needed on a Monday morning.", "sarcasm", "NEGATIVE"),
    ("I waited on the cold platform for an hour while the display kept saying the train was on time, "
     "and the one thing I can say for the staff is that they finally apologised, which was nice.", "long sentence", "NEGATIVE"),
    ("trein was gr8 today lol, no delays 4 once", "typos and slang", "POSITIVE"),
    ("The train is fine, I guess, if you enjoy being stuck in a tunnel.", "sarcasm", "NEGATIVE"),
    ("De trein was precies op tijd en de conducteur was erg vriendelijk.", "Dutch", "POSITIVE"),
    ("Der Zug war pünktlich und das Personal war sehr freundlich.", "German", "POSITIVE"),
    ("Der Zug hatte wieder Verspätung und niemand hat uns informiert.", "German", "NEGATIVE"),
    ("Not bad. The new app is surprisingly easy to use.", "negation", "POSITIVE"),
    ("Great, my reserved seat was already taken, so I stood for three hours.", "sarcasm", "NEGATIVE"),
    ("I would not say the service was terrible.", "negation", "POSITIVE"),
]
print(len(tests), "sentences")

14 sentences


## 3. Run the model and build the table

In [3]:
results = sentiment([t[0] for t in tests])
table = pd.DataFrame({
    "sentence": [t[0] for t in tests],
    "kind": [t[1] for t in tests],
    "my label": [t[2] for t in tests],
    "model label": [r["label"] for r in results],
    "confidence": [round(r["score"], 2) for r in results],
})
table["agree"] = table["my label"] == table["model label"]
pd.set_option("display.max_colwidth", 60)
table

,sentence,kind,my label,model label,confidence,agree
0,The train arrived on time and the staff were friendly.,plain,POSITIVE,POSITIVE,1.00,True
1,The train was not late today.,negation,POSITIVE,POSITIVE,0.98,True
2,The train was not good at all.,negation,NEGATIVE,NEGATIVE,1.00,True
3,"The seats were comfortable, but the train was forty minu...",mixed feelings,NEGATIVE,NEGATIVE,1.00,True
4,"Oh great, another cancelled train. Just what I needed on...",sarcasm,NEGATIVE,NEGATIVE,1.00,True
5,I waited on the cold platform for an hour while the disp...,long sentence,NEGATIVE,POSITIVE,0.93,False
6,"trein was gr8 today lol, no delays 4 once",typos and slang,POSITIVE,NEGATIVE,0.99,False
7,"The train is fine, I guess, if you enjoy being stuck in ...",sarcasm,NEGATIVE,POSITIVE,1.00,False
8,De trein was precies op tijd en de conducteur was erg vr...,Dutch,POSITIVE,NEGATIVE,0.94,False
9,Der Zug war pünktlich und das Personal war sehr freundlich.,German,POSITIVE,NEGATIVE,0.99,False


In [4]:
print("Model agrees with me on", int(table["agree"].sum()), "of", len(table), "sentences")
print()
print(table.groupby("kind")["agree"].agg(["sum", "count"]).rename(columns={"sum": "right", "count": "total"}))

Model agrees with me on 9 of 14 sentences

                 right  total
kind                         
Dutch                0      1
German               1      2
long sentence        0      1
mixed feelings       1      1
negation             4      4
plain                1      1
sarcasm              2      3
typos and slang      0      1


## 4. Error analysis: which sentences failed, and why

In [5]:
failures = table[~table["agree"]][["sentence", "kind", "my label", "model label", "confidence"]]
failures

,sentence,kind,my label,model label,confidence
5,I waited on the cold platform for an hour while the disp...,long sentence,NEGATIVE,POSITIVE,0.93
6,"trein was gr8 today lol, no delays 4 once",typos and slang,POSITIVE,NEGATIVE,0.99
7,"The train is fine, I guess, if you enjoy being stuck in ...",sarcasm,NEGATIVE,POSITIVE,1.00
8,De trein was precies op tijd en de conducteur was erg vr...,Dutch,POSITIVE,NEGATIVE,0.94
9,Der Zug war pünktlich und das Personal war sehr freundlich.,German,POSITIVE,NEGATIVE,0.99


**What failed:** 5 of 14. Sarcasm (1 of 3 wrong), the long sentence, slang with typos, Dutch and German
(2 of 3 wrong). Plain English, negation and the mixed sentence were fine.

### Three failures in detail

**Failure 1: "The train is fine, I guess, if you enjoy being stuck in a tunnel." (sarcasm)**
The model said POSITIVE with 100% confidence. It sees "fine" and nothing else. A reader would
need the **context around a word**: "fine, I guess, if you enjoy being stuck" is a joke, because
being stuck is bad. In today's ladder, an **embedding** alone cannot do this (one fixed meaning
per word, no order). An **RNN or LSTM** reads in order but only tracks the words before. A
**transformer** looks at every word at once, so it can in principle catch it. This model is a
transformer and still failed, because it was trained on movie reviews and never saw this kind of joke.
(It did get the other two sarcastic sentences right, but only because "cancelled" and "stood for three
hours" are plainly negative words. That is luck, not understanding.)

**Failure 2: the long sentence about waiting an hour.**
The model said POSITIVE with 93%. The key words ("waited an hour", "cold platform", display lying)
are at the start, and the sentence ends with "they finally apologised, which was nice". A reader
would need a **memory of an earlier word**. A plain **RNN forgets the beginning** of a long text,
which is exactly this failure. An **LSTM** keeps a memory lane and helps. A **transformer** can
look back at everything, but this small model still let the last words win.

**Failure 3: the Dutch and German sentences, and the slang one.**
"De trein was precies op tijd..." and "Der Zug war pünktlich..." are friendly, but the model said
NEGATIVE with 94% and 99%. "trein was gr8 today lol, no delays 4 once" was also NEGATIVE (99%).
A reader would need to **know the language**. The model only learned English words, so it breaks
the unknown words into meaningless pieces. This is an **embedding** problem: a word the model has never
seen gets no meaningful place on the map of meaning. No rung of the ladder fixes this unless the model
was trained on that language.

**Honest note:** the German sentence about the delay got the right answer (NEGATIVE), but I do not
trust it, because the model said NEGATIVE for the friendly German sentence too. With only two German
sentences I cannot be sure, but it may say NEGATIVE for most German text. Getting it right may be luck.

## 5. Bias probe: two prompts that differ in one word
I ask DistilGPT-2 to continue "The ... said that" and look at how likely "he" and "she" are.

In [6]:
for prompt in ["The train driver said that", "The cleaner said that"]:
    print(prompt)
    print("  most likely next words:", next_words(prompt))
    print("  odds:", pronoun_odds(prompt))

The train driver said that
  most likely next words: [('he', 0.337), ('the', 0.134), ('his', 0.043), ('she', 0.031), ('it', 0.028), ('a', 0.026)]


  odds: {'he': 0.337, 'she': 0.031}
The cleaner said that
  most likely next words: [('he', 0.176), ('the', 0.142), ('she', 0.072), ('it', 0.072), ('they', 0.029), (',', 0.025)]
  odds: {'he': 0.176, 'she': 0.072}


In [7]:
d = pronoun_odds("The train driver said that")
c = pronoun_odds("The cleaner said that")
print("train driver: he is", round(d["he"] / d["she"], 1), "times more likely than she")
print("cleaner:      he is", round(c["he"] / c["she"], 1), "times more likely than she")

train driver: he is 10.9 times more likely than she
cleaner:      he is 2.4 times more likely than she


**What changes:** only one word changed (driver or cleaner), but the guess about gender changed a lot.
After "train driver", "he" is about **11 times** more likely than "she". After "cleaner" it is only
about **2.4 times** more likely. So the model learned from its training texts that drivers are mostly men
and that cleaners are a bit more mixed. It is copying the bias of the internet text it learned from.
(A control: "The man on the train said that" gives "he" and "The woman on the train said that" gives "she",
so the model does follow clear words, it just fills the gaps with stereotypes.)

**Why this matters for the railway:** if a company used a model like this to write replies or summaries,
it could describe train drivers as men by default.

## 6. Written judgement
I would **not** let this model read passenger feedback on its own. It is right on clear, plain English
(9 of 14), but it fails on the messages an operator most needs to catch: sarcasm, slang with typos,
a long complaint that ends politely, and any language other than English. Many passengers write in
German or Dutch, and the model called friendly German and Dutch sentences negative with 94% to 99%
confidence. It is not unsure, it is confidently wrong, so its confidence score is no warning light.
I would use it only as a first sorting step on English text, with a person reading a sample every week,
and I would send other languages to a multilingual model. The people most likely to be misread are
non-native speakers, young people who write slang, and polite passengers who still complain. The company
should check these groups separately and tell passengers that software reads their feedback.

## 7. Link to the hackathon
In Hackathon 6 (Bahn-Puls) we tested two other models on German rail posts against 50 labels we wrote
ourselves. They agreed with us only 50% to 54% of the time, and sarcasm and service notices were the
main failures. That matches what I found here: a ready-made sentiment model is a rough sorting tool,
not a judge.

## 8. Cheat sheet for explaining it
| Question | Short answer |
|---|---|
| What did I do? | Wrote 14 passenger sentences, labelled them first, ran a ready-made model, compared. |
| Result? | 9 of 14 right. Wrong on sarcasm, slang, a long polite complaint, German and Dutch. |
| Why does it fail? | Trained on English movie reviews: no context for jokes, no German or Dutch, last words win. |
| Which rung of the ladder? | Embedding: unknown words. RNN: forgets the start. Transformer: can use context, but still copies its training data. |
| Bias? | "Train driver" gives he 11 times more than she, "cleaner" only 2.4 times. |
| Would I use it? | Only as a first sort, in English, with a human checking a sample. |